In [5]:
import os

import pandas as pd
import plotly.graph_objects as go
import requests
from IPython.display import HTML, display

try:
	from dotenv import load_dotenv

	load_dotenv()
except Exception:
	pass



def load_env_var_from_file(var_name: str) -> str:
	value = os.getenv(var_name, "").strip()
	if value:
		return value

	for candidate in (".env", "../.env"):
		if not os.path.exists(candidate):
			continue

		for encoding in ("utf-8", "utf-8-sig", "cp1252", "latin-1"):
			try:
				with open(candidate, "r", encoding=encoding) as handle:
					for raw_line in handle:
						line = raw_line.strip()
						if not line or line.startswith("#") or "=" not in line:
							continue
						key, raw_value = line.split("=", 1)
						if key.strip() == var_name:
							return raw_value.strip().strip('"').strip("'")
			except UnicodeDecodeError:
					continue
	return ""



def fetch_fred_series(series_id: str, api_key: str) -> pd.Series:
	if not api_key:
		raise ValueError("FRED_API_KEY is required to fetch data from the FRED API")

	url = (
		"https://api.stlouisfed.org/fred/series/observations"
		f"?series_id={series_id}&api_key={api_key}&file_type=json"
	)
	response = requests.get(url, timeout=30)
	response.raise_for_status()
	payload = response.json()
	observations = pd.DataFrame(payload.get("observations", []))

	if observations.empty:
		raise ValueError(f"No observations returned for {series_id}")

	observations["date"] = pd.to_datetime(observations["date"])
	observations["value"] = pd.to_numeric(observations["value"], errors="coerce")
	series = observations.dropna(subset=["value"]).set_index("date")["value"].sort_index()
	series.name = series_id
	return series


fred_api_key = load_env_var_from_file("FRED_API_KEY")
hy_oas = fetch_fred_series("BAMLH0A0HYM2", fred_api_key)

fig = go.Figure()
fig.add_trace(
	go.Scatter(
		x=hy_oas.index,
		y=hy_oas.values,
		mode="lines",
		name="High Yield OAS",
		line=dict(color="#1d4ed8", width=2),
	)
)
fig.update_layout(
	title="BAMLH0A0HYM2 High Yield Option-Adjusted Spread",
	xaxis_title="Date",
	yaxis_title="Percent",
	template="plotly_white",
	margin=dict(l=40, r=20, t=60, b=40),
)

display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False)))